# Bài 3: Naive Bayes - phần thực hành

**Họ tên:** Huỳnh Hữu Phúc  
**MSSV:** 2474802010314  
**Dữ liệu:** `data/tin_nhan.csv`, `data/sinh_vien.csv`  
**Cách chạy:** chạy toàn bộ cell với thư mục làm việc là thư mục `03. Naive_Bayes`.

Năm cell mã giữ theo mẫu trong PDF Lab 3. Chỉ sửa các ký tự in sai khi trích PDF (dấu trừ, dấu nháy và `token_pattern`) để Python chạy được.

## Bước 1. Định lý Bayes bằng tay

Tính hậu nghiệm từ tiên nghiệm và likelihood; kiểm tra lại bằng phép đếm.

In [1]:
# -*- coding: utf-8 -*-
"""Bước 1: tính định lý Bayes bằng tay, không cần thư viện học máy.

Chú thích viết tiếng Việt có dấu, phần in ra màn hình viết không dấu.
"""


def bayes(tien_nghiem, kha_nang, kha_nang_nguoc):
    """Tra ve xac suat hau nghiem.

    tien_nghiem : P(A), niem tin truoc khi co bang chung
    kha_nang : P(B | A), bang chung xuat hien khi A dung
    kha_nang_nguoc : P(B | khong A), bang chung xuat hien khi A sai
    """
    tu_so = kha_nang * tien_nghiem
    mau_so = tu_so + kha_nang_nguoc * (1 - tien_nghiem)
    return tu_so / mau_so


print("Vi du 1: xet nghiem benh")
print(" Ty le mac benh 1 phan tram, xet nghiem dung 99 phan tram.")
p = bayes(tien_nghiem=0.01, kha_nang=0.99, kha_nang_nguoc=0.01)
print(f" Duong tinh thi kha nang co benh that = {p:.4f}")
print()

# Giữ nguyên chất lượng xét nghiệm, chỉ đổi tỷ lệ mắc bệnh
print("Cung xet nghiem do, doi ty le mac benh:")
print(" Ty le mac benh Hau nghiem khi duong tinh")
for ty_le in [0.001, 0.01, 0.05, 0.10, 0.50]:
    print(f" {ty_le:6.3f} {bayes(ty_le, 0.99, 0.01):.4f}")
print()

print("Vi du 2: hai day chuyen san xuat")
print(" Day A lam 80 phan tram san pham, ty le loi 1 phan tram.")
print(" Day B lam 20 phan tram, ty le loi 5 phan tram.")
# Ở đây "A đúng" nghĩa là sản phẩm đến từ dây B
p_b = bayes(tien_nghiem=0.20, kha_nang=0.05, kha_nang_nguoc=0.01)
print(f" Nhat duoc san pham loi, kha nang no tu day B = {p_b:.4f}")
print()

print("Kiem lai bang cach dem tren 1000 san pham:")
loi_a = 800 * 0.01
loi_b = 200 * 0.05
print(f" Day A: 800 san pham, {loi_a:.0f} cai loi")
print(f" Day B: 200 san pham, {loi_b:.0f} cai loi")
tong_loi = loi_a + loi_b
print(f" Ty le tu day B = {loi_b} / {tong_loi} = {loi_b / tong_loi:.4f}")


Vi du 1: xet nghiem benh
 Ty le mac benh 1 phan tram, xet nghiem dung 99 phan tram.
 Duong tinh thi kha nang co benh that = 0.5000

Cung xet nghiem do, doi ty le mac benh:
 Ty le mac benh Hau nghiem khi duong tinh
  0.001 0.0902
  0.010 0.5000
  0.050 0.8390
  0.100 0.9167
  0.500 0.9900

Vi du 2: hai day chuyen san xuat
 Day A lam 80 phan tram san pham, ty le loi 1 phan tram.
 Day B lam 20 phan tram, ty le loi 5 phan tram.
 Nhat duoc san pham loi, kha nang no tu day B = 0.5556

Kiem lai bang cach dem tren 1000 san pham:
 Day A: 800 san pham, 8 cai loi
 Day B: 200 san pham, 10 cai loi
 Ty le tu day B = 10.0 / 18.0 = 0.5556


**Nhận xét:** Xét nghiệm dương tính cho hậu nghiệm 0,5000 dù độ nhạy là 0,99 vì tỷ lệ mắc bệnh ban đầu chỉ 0,01. Với dây chuyền B, hậu nghiệm là 0,5556.

## Bước 2. Biến văn bản thành túi từ

Dùng `CountVectorizer` với `token_pattern=r"(?u)\S+"` để giữ các từ một ký tự.

In [2]:
# -*- coding: utf-8 -*-
"""Bước 2: đọc kho tin nhắn và biến văn bản thành bảng số."""

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer


df = pd.read_csv("data/tin_nhan.csv")

print("Kich thuoc bang:", df.shape)
print()
print("So tin theo nhan (1 la rac, 0 la thuong):")
print(df["la_rac"].value_counts())
print()
print("Ty le tin rac:", round(df["la_rac"].mean(), 4))
print()

print("Ba tin nhan dau tien:")
for i in range(3):
    print(f" [{df.la_rac[i]}] {df.noi_dung[i]}")
print()

# token_pattern mặc định của scikit-learn bỏ mất từ một ký tự,
# nên ta chỉ định tách từ bằng khoảng trắng cho đơn giản.
vec = CountVectorizer(token_pattern=r"(?u)\S+")
X = vec.fit_transform(df["noi_dung"])

print("Kich thuoc bang dem tu:", X.shape)
print("So tu trong tu dien:", len(vec.vocabulary_))
print()

# Đếm tổng số lần xuất hiện của từng từ, tách theo hai lớp
dem = pd.DataFrame(X.toarray(), columns=vec.get_feature_names_out())
dem["la_rac"] = df["la_rac"]
tong = dem.groupby("la_rac").sum().T
tong.columns = ["thuong", "rac"]

print("Muoi tu xuat hien nhieu nhat trong tin rac:")
print(tong.sort_values("rac", ascending=False).head(10))
print()
print("Muoi tu xuat hien nhieu nhat trong tin thuong:")
print(tong.sort_values("thuong", ascending=False).head(10))


Kich thuoc bang: (300, 2)

So tin theo nhan (1 la rac, 0 la thuong):
la_rac
0    180
1    120
Name: count, dtype: int64

Ty le tin rac: 0.4

Ba tin nhan dau tien:
 [0] cáo gửi cáo thầy nhóm cáo gửi có cho
 [1] miễn giảm ngay hạn miễn tay nhé thưởng cho nhanh
 [0] cho bạn mai gửi nhóm bài chiều

Kich thuoc bang dem tu: (300, 40)
So tu trong tu dien: 40

Muoi tu xuat hien nhieu nhat trong tin rac:
       thuong  rac
tay         3   48
ngay        4   47
nhanh       9   44
chót        6   43
giảm        6   42
lúc        39   42
quà         8   41
click       5   40
phí         6   40
miễn        4   39

Muoi tu xuat hien nhieu nhat trong tin thuong:
       thuong  rac
thầy       74    7
bài        73    4
cáo        72    8
chiều      68    8
ạ          65    7
tài        64   11
tập        63    6
gửi        61   10
lớp        60    2
nhóm       58    7


**Nhận xét:** Kho có 300 tin, gồm 180 tin thường và 120 tin rác. Túi từ có 40 cột từ vựng.

## Bước 3. Bộ lọc tin nhắn rác

Dùng đúng trình tự trong PDF: biến đổi 300 tin rồi chia dữ liệu với `random_state=17`, `stratify=y`.

In [3]:
# -*- coding: utf-8 -*-
"""Bước 3: huấn luyện bộ lọc thư rác và chấm điểm nó."""

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB


df = pd.read_csv("data/tin_nhan.csv")
vec = CountVectorizer(token_pattern=r"(?u)\S+")
X = vec.fit_transform(df["noi_dung"])
y = df["la_rac"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
print("So tin de hoc :", X_train.shape[0])
print("So tin de kiem tra:", X_test.shape[0])
print()

mo_hinh = MultinomialNB(alpha=1.0)
mo_hinh.fit(X_train, y_train)
y_pred = mo_hinh.predict(X_test)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print("Ma tran nham lan")
print(f" TN = {tn:2d} thu thuong duoc giu dung")
print(f" FP = {fp:2d} thu thuong bi xoa nham")
print(f" FN = {fn:2d} thu rac lot luoi")
print(f" TP = {tp:2d} thu rac bi chan dung")
print()
print(f"Accuracy = {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision = {precision_score(y_test, y_pred):.4f}")
print(f"Recall = {recall_score(y_test, y_pred):.4f}")
print()

# Thử bộ lọc trên vài tin nhắn tự gõ
tin_moi = [
    "trung thuong khuyen mai nhanh tay click ngay",
    "hop nhom bao cao chieu mai phong 305",
    "ban gui giup em tai lieu bai tap nhe",
]
X_moi = vec.transform(tin_moi)
p_rac = mo_hinh.predict_proba(X_moi)[:, 1]
print("Thu tren ba tin nhan tu go:")
for tin, p in zip(tin_moi, p_rac):
    nhan = "RAC" if p >= 0.5 else "thuong"
    print(f" [{nhan:6s}] p = {p:.4f} <- {tin}")


So tin de hoc : 225
So tin de kiem tra: 75

Ma tran nham lan
 TN = 45 thu thuong duoc giu dung
 FP =  0 thu thuong bi xoa nham
 FN =  4 thu rac lot luoi
 TP = 26 thu rac bi chan dung

Accuracy = 0.9467
Precision = 1.0000
Recall = 0.8667

Thu tren ba tin nhan tu go:
 [RAC   ] p = 0.9992 <- trung thuong khuyen mai nhanh tay click ngay
 [thuong] p = 0.0983 <- hop nhom bao cao chieu mai phong 305
 [thuong] p = 0.0719 <- ban gui giup em tai lieu bai tap nhe


**Nhận xét:** Trong 75 tin kiểm tra: TN=45, FP=0, FN=4, TP=26. Accuracy=0,9467, precision=1,0000, recall=0,8667. Không xóa nhầm tin thường, nhưng còn lọt bốn tin rác.

## Bước 4. Vấn đề số không và làm trơn Laplace

Tự đếm sáu tin nhắn, so sánh `alpha=0` và `alpha=1`.

In [4]:
# -*- coding: utf-8 -*-
"""Bước 4: tự cài Naive Bayes trên một kho nhỏ để thấy vấn đề số không.

Đoạn này cố ý KHÔNG dùng scikit-learn, vì thư viện đã làm trơn sẵn nên
sinh viên không nhìn thấy chỗ hỏng.
"""

from collections import Counter


# Kho tin nhắn tí hon, viết thẳng vào mã cho dễ theo dõi
kho = [
    ("trung thuong ngay", 1),
    ("khuyen mai giam gia", 1),
    ("trung thuong click", 1),
    ("hop nhom chieu mai", 0),
    ("bai tap nhom nop mai", 0),
    ("thay gui tai lieu", 0),
]
tu_dien = sorted({t for cau, _ in kho for t in cau.split()})
V = len(tu_dien)
print(f"Tu dien co {V} tu:")
# In thành nhiều dòng, mỗi dòng 9 từ, để không tràn ra ngoài khung
for i in range(0, V, 9):
    print(" " + " ".join(tu_dien[i:i + 9]))
print()

# Đếm số lần mỗi từ xuất hiện trong từng lớp
dem = {0: Counter(), 1: Counter()}
tong = {0: 0, 1: 0}
for cau, nhan in kho:
    for t in cau.split():
        dem[nhan][t] += 1
        tong[nhan] += 1

print("So lan xuat hien cua tung tu")
print(f" {'tu':12s} {'rac':>5s} {'thuong':>7s}")
for t in tu_dien:
    print(f" {t:12s} {dem[1][t]:5d} {dem[0][t]:7d}")
print()


def xac_suat(tu, nhan, alpha):
    """P(tu | nhan) co lam tron Laplace voi he so alpha."""
    return (dem[nhan][tu] + alpha) / (tong[nhan] + alpha * V)


def diem(cau, nhan, alpha):
    """Tich cua xac suat tien nghiem va cac xac suat tu."""
    p = 0.5  # hai lop can bang nen tien nghiem bang nhau
    for t in cau.split():
        if t in tu_dien:
            p *= xac_suat(t, nhan, alpha)
    return p


cau_thu = "trung thuong bai tap"
print(f"Tin nhan thu: \"{cau_thu}\"")
print("Tu 'bai' chi xuat hien o lop thuong, chua bao gio o lop rac.")
print()
for alpha in [0.0, 1.0]:
    d_rac = diem(cau_thu, 1, alpha)
    d_thuong = diem(cau_thu, 0, alpha)
    ket = "rac" if d_rac > d_thuong else "thuong"
    print(f"alpha = {alpha}")
    print(f" diem lop rac = {d_rac:.10f}")
    print(f" diem lop thuong = {d_thuong:.10f}")
    print(f" ket luan = {ket}")
    print()
print("Khong lam tron, CA HAI diem deu bang 0 nen mo hinh te liet.")
print("Lop rac bang 0 vi tu 'bai' chua tung xuat hien o lop rac.")
print("Lop thuong bang 0 vi tu 'trung' chua tung xuat hien o lop thuong.")
print("Co lam tron, diem lop rac lon hon va mo hinh ket luan dung.")


Tu dien co 18 tu:
 bai chieu click gia giam gui hop khuyen lieu
 mai ngay nhom nop tai tap thay thuong trung

So lan xuat hien cua tung tu
 tu             rac  thuong
 bai              0       1
 chieu            0       1
 click            1       0
 gia              1       0
 giam             1       0
 gui              0       1
 hop              0       1
 khuyen           1       0
 lieu             0       1
 mai              1       2
 ngay             1       0
 nhom             0       2
 nop              0       1
 tai              0       1
 tap              0       1
 thay             0       1
 thuong           2       0
 trung            2       0

Tin nhan thu: "trung thuong bai tap"
Tu 'bai' chi xuat hien o lop thuong, chua bao gio o lop rac.

alpha = 0.0
 diem lop rac = 0.0000000000
 diem lop thuong = 0.0000000000
 ket luan = thuong

alpha = 1.0
 diem lop rac = 0.0000073212
 diem lop thuong = 0.0000021656
 ket luan = rac

Khong lam tron, CA HAI diem deu bang 0 nen mo 

**Nhận xét:** Không làm trơn thì cả hai tích bằng 0 và mất khả năng phân biệt. Làm trơn Laplace giữ xác suất từng từ khác 0; với tin thử, điểm lớp rác lớn hơn.

## Bước 5. GaussianNB cho số liệu liên tục

So sánh GaussianNB với hồi quy logistic trên cùng tập sinh viên.

In [5]:
# -*- coding: utf-8 -*-
"""Bước 5: Naive Bayes cho số liệu liên tục, đặt cạnh hồi quy logistic."""

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB


df = pd.read_csv("data/sinh_vien.csv")
X = df[["gio_on", "diem_giua_ky"]]
y = df["qua_mon"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# Hai mô hình, cùng dữ liệu, cùng cách chia, nên so sánh mới công bằng
for ten, mo_hinh in [("Naive Bayes ", GaussianNB()),
                     ("Hoi quy logistic", LogisticRegression())]:
    mo_hinh.fit(X_train, y_train)
    acc = accuracy_score(y_test, mo_hinh.predict(X_test))
    print(f"{ten}: accuracy = {acc:.4f}")
print()

gnb = GaussianNB().fit(X_train, y_train)

# theta_ la trung binh, var_ la phuong sai, moi dong mot lop
print("Nhung gi Naive Bayes hoc duoc")
print(f" {'':16s} {'gio_on':>10s} {'diem_giua_ky':>14s}")
for i, ten_lop in enumerate(["Lop 0 (rot) ", "Lop 1 (qua) "]):
    tb = gnb.theta_[i]
    dl = gnb.var_[i] ** 0.5
    print(f" {ten_lop} trung binh {tb[0]:10.2f} {tb[1]:14.2f}")
    print(f" {'':12s} do lech {dl[0]:10.2f} {dl[1]:14.2f}")
print()
ty_le = ", ".join(f"{float(v):.4f}" for v in gnb.class_prior_)
print(f"Ty le tung lop trong tap hoc: {ty_le}")
print()

# Dự đoán cho vài sinh viên giả định
ban_moi = pd.DataFrame({"gio_on": [5, 13, 20, 28],
                       "diem_giua_ky": [4.0, 6.0, 7.0, 9.0]})
p = gnb.predict_proba(ban_moi)[:, 1]
print("Du doan cho bon ban moi:")
print(f" {'gio on':>8s} {'diem GK':>9s} {'P(qua mon)':>12s}")
for (_, r), pi in zip(ban_moi.iterrows(), p):
    print(f" {r.gio_on:8.0f} {r.diem_giua_ky:9.1f} {pi:12.4f}")


Naive Bayes : accuracy = 0.9000
Hoi quy logistic: accuracy = 0.9000

Nhung gi Naive Bayes hoc duoc
                      gio_on   diem_giua_ky
 Lop 0 (rot)  trung binh       8.18           3.99
              do lech       5.29           1.92
 Lop 1 (qua)  trung binh      20.03           6.70
              do lech       5.81           1.50

Ty le tung lop trong tap hoc: 0.4111, 0.5889

Du doan cho bon ban moi:
   gio on   diem GK   P(qua mon)
        5       4.0       0.0137
       13       6.0       0.6550
       20       7.0       0.9855
       28       9.0       0.9999


**Nhận xét:** Hai mô hình cùng đạt accuracy=0,9000. GaussianNB còn cho biết trung bình giờ ôn của lớp rớt/qua môn lần lượt là 8,18/20,03 giờ.